# MTG strength: GPU training on Google Colab

Interactive alternative to the Kaggle job (`gpu/kaggle/job.py`); see `docs/GPU.md`.
Runtime > Change runtime type > **T4 GPU**, then run the cells in order. Keep the tab open:
a free Colab runtime stops after ~90 min idle or ~12 h in total.

Secrets (key icon on the left): add `HF_TOKEN` (a Hugging Face token whose account accepted the
EmbeddingGemma licence) and enable notebook access.

In [ ]:
BRANCH = "main"
NAME = "ft_embeddinggemma2_k4_gpu"
SAVE_TO_DRIVE = True   # caches and results survive a disconnect

In [ ]:
import os, subprocess
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORE = "/content/drive/MyDrive/mtg_strength_gpu"
else:
    STORE = "/content/store"
os.makedirs(STORE, exist_ok=True)
!nvidia-smi

In [ ]:
%cd /content
!rm -rf repo && git clone --depth 1 --branch {BRANCH} https://github.com/ContrattoAlessandro/mtg_strenght repo
%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -U transformers sentence-transformers

Phase 2 fine-tune. The hidden-state cache and the per-epoch checkpoints go to `STORE/cache`,
so after a disconnect, re-running the cells above and this one resumes the run.

In [ ]:
!python scripts/phase2.py train --encoder google/embeddinggemma-2 --top-k 4 \
    --name {NAME} --max-epochs 6 --seeds 0 1 2 \
    --device cuda --cache-dir {STORE}/cache --save-dir models/phase2

Copy what the run produced (repo-relative paths) to `STORE/results`, and zip it for download.
Unzip `results.zip` at the repo root on your machine.

In [ ]:
files = subprocess.run("git ls-files --modified --others --exclude-standard out models;"
                       " git ls-files --others --ignored --exclude-standard out models",
                       shell=True, capture_output=True, text=True).stdout.split()
print("\n".join(sorted(set(files))))
!mkdir -p {STORE}/results && cp --parents {" ".join(sorted(set(files)))} {STORE}/results/
!cd {STORE}/results && zip -qr ../results.zip . && echo wrote {STORE}/results.zip